In [47]:
from typing import TypedDict
from langgraph.graph import StateGraph , START , END
from dotenv import load_dotenv
load_dotenv()
from langchain_google_genai import ChatGoogleGenerativeAI

In [48]:
model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0.7
)

In [49]:
# create state 
class LLMState(TypedDict):
   topic: str
   eassy: str
   summary: str
   

In [50]:
def llm_te(state: LLMState) -> LLMState:

    question = state["topic"]

    prompt = f"Write an essay on the following topic : {question}"

    ans = model.invoke(prompt).content

    return {
        "eassy": ans
    }

In [51]:

def llm_es(state :LLMState) ->LLMState:
    """
    A simple function that takes a paragarph and returns an summary in 100 words using the LLM model.
    """
    eassy = state['eassy']
    prompt = f"write an summary on the following paragraph in 100 words: {eassy}"
    ans = model.invoke(prompt).content
   
    return {
        "summary" : ans
    }
  

In [52]:
# graph = StateGraph(LLMState)
# graph.add_node("llm_te", llm_te)
# graph.add_node("llm_es", llm_es)
# graph.add_edge(START, "llm_te")
# graph.add_edge('llm_te' , 'llm_es')
# graph.add_edge("llm_es", END)

# workflow = graph.compile()

In [53]:
# initial_state = {"topic": "Agentic ai mcp importance?"}
# final_state = workflow.invoke(initial_state)
# print(final_state["summary"][0]["text"])  # Output: "The capital of France is Paris."

                 llm_te
                   ↓
             Is essay long?
              /         \
            YES           NO
             ↓             ↓
         llm_es         END
             ↓
            END

In [59]:
def check_essay(state: LLMState):

    if len(state["eassy"]) > 1000:
        return "summarize"

    return "end"

In [60]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(LLMState)

graph.add_node("llm_te", llm_te)
graph.add_node("llm_es", llm_es)

graph.add_edge(START, "llm_te")

In [61]:
graph.add_conditional_edges(
    "llm_te",
    check_essay,
    {
        "summarize": "llm_es",
        "end": END
    }
)

In [62]:
graph.add_edge("llm_es", END)

workflow = graph.compile()

In [64]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END


# =================================
# STATE
# =================================

class LLMState(TypedDict):
    topic: str
    essay: str
    summary: str


# =================================
# NODE 1: Generate Essay
# =================================

def llm_te(state: LLMState) -> LLMState:

    topic = state["topic"]

    prompt = f"""
    Write an essay on the following topic
    in 600 words:

    {topic}
    """

    ans = model.invoke(prompt).content

    return {
        "essay": ans
    }


# =================================
# NODE 2: Generate Summary
# =================================

def llm_es(state: LLMState) -> LLMState:

    essay = state["essay"]

    prompt = f"""
    Write a summary of the following essay
    in 100 words:

    {essay}
    """

    ans = model.invoke(prompt).content

    return {
        "summary": ans
    }


# =================================
# ROUTING FUNCTION
# =================================

def check_essay(state: LLMState):

    if len(state["essay"]) > 1000:
        return "summarize"

    return "end"


# =================================
# GRAPH
# =================================

graph = StateGraph(LLMState)

graph.add_node("llm_te", llm_te)
graph.add_node("llm_es", llm_es)


# START → Essay
graph.add_edge(
    START,
    "llm_te"
)


# Conditional routing
graph.add_conditional_edges(
    "llm_te",
    check_essay,
    {
        "summarize": "llm_es",
        "end": END
    }
)


# Summary → END
graph.add_edge(
    "llm_es",
    END
)


# Compile
workflow = graph.compile()

In [66]:
result = workflow.invoke({
    "topic": "Artificial Intelligence",
    "essay": "",
    "summary": ""
})

print(result)

{'topic': 'Artificial Intelligence', 'essay': [{'type': 'text', 'text': '**Title:** The Cognition of Code: Navigating the Era of Artificial Intelligence\n\nArtificial Intelligence (AI) has swiftly transitioned from the realm of science fiction into the fabric of daily life. Once confined to theoretical mathematics and speculative literature, AI now powers the algorithms that curate our social media feeds, diagnose complex medical conditions, navigate autonomous vehicles, and generate creative works. This technological revolution represents a fundamental shift in human history: for the first time, humanity is creating systems that not only execute tasks but also learn, adapt, and make decisions independently. As AI continues its exponential growth, it brings with it a complex tapestry of unprecedented opportunities and profound ethical dilemmas.\n\nAt its core, artificial intelligence is the simulation of human intelligence by machine systems. Through the pillars of machine learning and

             ┌─────────┐
             │   User  │
             └────┬────┘
                  ↓
             ┌─────────┐
             │   LLM   │
             └────┬────┘
                  ↓
          Tool requested?
             /       \
           NO         YES
           ↓           ↓
         END       ToolNode
                       ↓
                  Tool executes
                       ↓
                  Tool result
                       ↓
                      LLM
                       ↓
                 Tool requested?
                    /      \
                  NO        YES
                  ↓          ↓
                 END      ToolNode

In [27]:
import os
from typing import Annotated, TypedDict
from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

load_dotenv()

# ==========================================
# 1. Define Tools
# ==========================================
@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

tools = [add]

# ==========================================
# 2. LLM with Tools Bound
# ==========================================
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0.7
)
llm_with_tools = llm.bind_tools(tools)

# ==========================================
# 3. State Definition
# ==========================================
class State(TypedDict):
    # add_messages reducer appends new messages to the existing list
    messages: Annotated[list, add_messages]

# ==========================================
# 4. Nodes & Routing Condition
# ==========================================
def llm_node(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {
        "messages": [response]
    }

tool_node = ToolNode(tools)

def should_cont(state: State):
    last_msg = state["messages"][-1]
    # Check if LLM requested a tool call
    print(last_msg)
    if last_msg.tool_calls:
        return "tools"
    return END

# ==========================================
# 5. Build Graph
# ==========================================
graph = StateGraph(State)

graph.add_node("llm", llm_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "llm")
graph.add_conditional_edges(
    "llm",
    should_cont,
    {
        "tools": "tools",
        END: END
    }
)
graph.add_edge("tools", "llm")

app = graph.compile()

# ==========================================
# 6. Run & Test
# ==========================================
result = app.invoke({
    "messages": [
        ("user", "What is 25 + 17?")
    ]
})

# Print final AI response
print(result["messages"][-1].content[0]["text"])


content=[] additional_kwargs={'function_call': {'name': 'add', 'arguments': '{"b": 17, "a": 25}'}, '__gemini_function_call_thought_signatures__': {'call_3685757': 'EmAKXgFpFH0TDadoVIp4SV3iXNSJEuH88VR1NCMVlq3UW5MsQCwF5wI/dyjHNrUTWJB2sTsVSxRWzxPPD49pcob+lJYC0mXRm2UaDb4fD+GUldoeMrnpUX9csvi/77IVDkY='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a1069d-0aa6-7862-8a97-64b6b4278688-0' tool_calls=[{'name': 'add', 'args': {'b': 17, 'a': 25}, 'id': 'call_3685757', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 62, 'output_tokens': 18, 'total_tokens': 80, 'input_token_details': {'cache_read': 0}}
content=[{'type': 'text', 'text': '25 + 17 = 42', 'extras': {'signature': 'EmAKXgFpFH0T7XbxMOwpRzdJ0Nbl/3vJLdQgnFjYrRP5B4E7bF4456z84LKF6tEAb0FwHg9m1srSsmDiZX6PBehvC93uSklGIYvzb5JeMgZ3TSupbmJpl3YCJJgtg4Kicuk='}}] additional_kwargs={} response_metadata={'finish_reason':

# MCP (Model Context Protocol) — Quick Revision Notes

## 1. What is MCP?

**MCP = Model Context Protocol**

A standard protocol that allows an AI application/agent to communicate with external **tools, data, and services**.

> **MCP = standardized bridge between an AI application and external capabilities.**

MCP is **not**:
- An LLM
- An agent framework
- A database
- A replacement for LangGraph

---

## 2. Main Components

```text
User
  ↓
AI Application / Agent
  ↓
LLM
  ↓
MCP Client
  ↓ MCP
MCP Server
  ↓
Tools / Resources / Prompts
```

### MCP Host
The application where the AI/agent runs.

Examples:
- AI application
- IDE agent
- LangGraph application

### MCP Client
The component inside the application that communicates with an MCP server.

### MCP Server
A server/process that exposes capabilities such as tools.

```text
MCP Server
├── Tool 1
├── Tool 2
├── Tool 3
├── Resources
└── Prompts
```

---

# 3. How Does the LLM Know Which Tools Exist?

This is the key concept.

The application connects an **MCP Client** to an **MCP Server**.

The client can discover the server's available tools.

```text
MCP Client
     │
     │ "What tools do you have?"
     ▼
MCP Server
     │
     │ Tool definitions
     ▼
MCP Client
```

The server provides information such as:

```text
Tool: dictionary

Description:
Look up the meaning of a word.

Input:
word → string
```

The application/agent makes these tool definitions available to the LLM.

So the LLM effectively knows:

```text
Available tools:

dictionary(word)
translate(text, language)
grammar_check(text)
```

### Important

The LLM does **not** automatically know every tool in the world.

It only knows the tools that the application makes available to it through tool definitions/schemas.

---

# 4. How Does the LLM Choose a Tool?

Suppose the user asks:

> "What does ubiquitous mean?"

The LLM sees:

```text
Available tools:

dictionary(word)
translate(text, language)
grammar_check(text)
```

It decides:

```text
"I need the dictionary tool."
```

It generates a **tool call**:

```text
dictionary(
    word="ubiquitous"
)
```

The LLM **does not execute the tool**.

It only requests the tool call.

---

# 5. Complete MCP Tool Flow

```text
USER
  ↓
LLM
  ↓
Decides a tool is needed
  ↓
Tool Call
  ↓
Application / Agent Runtime
  ↓
MCP Client
  ↓
MCP Server
  ↓
Execute Tool
  ↓
Tool Result
  ↓
MCP Client
  ↓
Application / Agent
  ↓
LLM
  ↓
Final Answer
  ↓
USER
```

---

# 6. Example

User:

> "What does ubiquitous mean?"

### Step 1 — LLM

```text
LLM → Need dictionary tool
```

### Step 2 — Tool call

```text
dictionary("ubiquitous")
```

### Step 3 — Application

The application receives the tool call and uses the MCP client.

```text
Application
    ↓
MCP Client
```

### Step 4 — MCP

```text
MCP Client
    ↓
MCP Server
    ↓
dictionary()
```

### Step 5 — Result

```text
"Present or found everywhere"
```

### Step 6 — Back to LLM

```text
MCP Server
    ↓
MCP Client
    ↓
Application
    ↓
LLM
```

The LLM then generates:

```text
"Ubiquitous means present or found everywhere."
```

---

# 7. Multiple MCP Servers

An application can connect to multiple MCP servers.

```text
                         LLM
                          │
                          ▼
                    Agent / App
                          │
                     MCP Client(s)
                    /       |       \
                   ↓        ↓        ↓
              MCP Server MCP Server MCP Server
              Language    Database     Web
                 │           │          │
              Tools       Tools      Tools
```

For example:

```text
Language MCP
├── dictionary()
├── translate()
└── grammar_check()

Database MCP
├── get_user()
└── get_progress()

Web MCP
├── search_web()
└── search_news()
```

The LLM chooses the appropriate available tool based on the user's request and the tool descriptions.

---

# 8. MCP vs Normal LangChain Tool

### Normal LangChain Tool

```text
LangGraph
   ↓
LLM
   ↓
ToolNode
   ↓
Python Function
```

Example:

```python
@tool
def add(a: int, b: int):
    return a + b
```

The tool is part of your application.

### MCP Tool

```text
LangGraph / App
      ↓
  MCP Client
      ↓
     MCP
      ↓
  MCP Server
      ↓
   MCP Tool
```

The tool can live outside the agent application and be accessed through the standardized MCP protocol.

---

# 9. MCP + LangGraph

LangGraph handles the **agent workflow/loop**.

MCP handles the **standardized connection to external capabilities**.

```text
User
 ↓
LangGraph
 ↓
LLM
 ↓
Tool Decision
 ↓
MCP Client
 ↓
MCP Server
 ↓
MCP Tool
 ↓
Result
 ↓
LLM
 ↓
Final Answer
```

If another tool is needed:

```text
LLM
 ↓
Another Tool Call
 ↓
MCP Client
 ↓
MCP Server
 ↓
Result
 ↓
LLM
```

This is the **agent loop**.

---

# 10. Most Important Mental Model

Remember these roles:

| Component | Role |
|---|---|
| **LLM** | Decides whether a tool is needed and which tool to call |
| **Agent / Application** | Orchestrates the interaction |
| **MCP Client** | Communicates with MCP servers |
| **MCP Server** | Exposes tools/capabilities |
| **MCP Tool** | Performs the actual operation |
| **LangGraph** | Controls the agent workflow/loop |
| **MCP** | Standard protocol for connecting applications to capabilities |

### One-line summary

> **LLM decides → App/Agent orchestrates → MCP Client communicates → MCP Server executes → Result returns → LLM responds.**

---

# 11. Learning Path

```text
22. Create MCP Server
        ↓
23. Create MCP Tool
        ↓
24. Create MCP Client
        ↓
25. Use MCP Tools in LangGraph
        ↓
      ⭐
Agent + MCP
```

The most important concept to understand before coding:

```text
             LLM
              │
       "I need tool X"
              ↓
        Application
              ↓
         MCP Client
              ↓
          MCP Server
              ↓
          Tool X
              ↓
           Result
              ↓
             LLM
              ↓
        Final Answer
```